In [ ]:
from dotenv import load_dotenv
import os

from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory

load_dotenv()
print(f"API_KEY: {os.getenv('API_KEY')}")
print(f"API_URL: {os.getenv('API_URL')}")
print(f"API_MODEL: {os.getenv('API_MODEL')}")

In [ ]:
from dataclasses import Field

from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate

import os

from pydantic import BaseModel, Field

api_key = os.getenv('DASHSCOPE_API_KEY')
api_url = os.getenv('DASHSCOPE_API_URL')
default_model = os.getenv('DEFAULT_LLM_MODEL')
default_temperature = os.getenv('API_TEMPERATURE',0)

# llm输出格式
class FinAnswer(BaseModel):
    answer: str = Field(description="答案选项")
    reason: str = Field(description="选择原因")

parser = PydanticOutputParser(pydantic_object=FinAnswer)

def build_llm(llm_model=default_model, **args):
    chat_model = ChatOpenAI(
        model=default_model,
        api_key=api_key,
        base_url=api_url,
        temperature=args['temperature'] if 'temperature' in args else default_temperature,
        **{k: v for k, v in args.items() if k!='temperature'})
    return chat_model
args = {"extra_body": {"enable_thinking": False}}
llm = build_llm('qwen-turbo', **args)

def call_llm(question:str):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "你是一个{role}。你的任务是根据用户的问题给出明确的答案。\n{response_instructions}"),
        ("human", "{question}")
    ])
    chain = prompt | llm | parser
    response = chain.invoke({
        "role": "专业的基金投研分析师",
        "response_instructions": parser.get_format_instructions(),
        "question": question})
    return response


In [ ]:
import json

import pandas as pd
fin_df = pd.read_csv('findata/val/finance_val.csv')

results = []
for row in fin_df.itertuples():
    test = {
        "question": row.question,
        "options": {
            "A": row.A,
            "B": row.B,
            "C": row.C,
            "D": row.D
        }
    }
    response:FinAnswer = call_llm(json.dumps(test, ensure_ascii=False))
    print(response)
    test['llm_answer'] = response.answer
    test['expect_answer'] = row.answer
    test['result'] = 1 if test['llm_answer'] == test['expect_answer'] else 0
    results.append(test)

df = pd.DataFrame(results)
df
